# 🧲 Physics-Informed Neural Network (PINN)
## Simple Harmonic Oscillator — Newton's 2nd Law

---

### The Physical System

A mass **m** attached to a spring with constant **k**. Newton's 2nd Law gives:

$$m\ddot{x} + kx = 0$$

The analytical solution is:

$$x(t) = A\cos(\omega t + \phi), \quad \omega = \sqrt{\frac{k}{m}}$$

### The Challenge

We have only **5 noisy measurements** of x(t). Can a neural network reconstruct the full trajectory?

- **Standard NN:** uses only data → fails with so few points
- **PINN:** uses data + physics law → recovers the true solution

### Loss Function

$$\mathcal{L} = \underbrace{\frac{1}{N_d}\sum(x_{NN} - x_{data})^2}_{\mathcal{L}_{data}} + \lambda\underbrace{\frac{1}{N_f}\sum(m\ddot{x}_{NN} + kx_{NN})^2}_{\mathcal{L}_{physics}}$$

## 📦 1. Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Dark plot style
plt.rcParams.update({
    'figure.dpi': 130, 'figure.facecolor': '#0d1117',
    'axes.facecolor': '#0d1117', 'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#e6edf3', 'xtick.color': '#e6edf3',
    'ytick.color': '#e6edf3', 'text.color': '#e6edf3',
    'grid.color': '#21262d', 'grid.linewidth': 0.6,
    'axes.grid': True, 'axes.spines.top': False,
    'axes.spines.right': False,
})
ACCENT = '#58a6ff'; WARM = '#f78166'; GREEN = '#56d364'
YELLOW = '#e3b341'; PURPLE = '#d2a8ff'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ PyTorch {torch.__version__} | Device: {device}')

## ⚙️ 2. Physical Parameters & Analytical Solution

In [ ]:
# ── Physical parameters ───────────────────────────────────────────────────────
m = 1.0          # mass [kg]
k = 4.0          # spring constant [N/m]
omega = np.sqrt(k / m)   # natural frequency [rad/s]
A = 1.0          # amplitude [m]
phi = 0.0        # initial phase [rad]
T = 2 * np.pi    # one full period [s]

print(f'Physical parameters:')
print(f'  m = {m} kg  |  k = {k} N/m')
print(f'  ω = √(k/m) = {omega:.4f} rad/s')
print(f'  Period T = 2π/ω = {2*np.pi/omega:.4f} s')

# ── Analytical solution ───────────────────────────────────────────────────────
# x(t) = A·cos(ωt + φ)
def x_exact(t):
    return A * np.cos(omega * t + phi)

# Time domain: 0 to 2π (one full period + a bit)
t_full = np.linspace(0, T, 1000)
x_full = x_exact(t_full)

# ── Sparse noisy observations (what we "measure") ─────────────────────────────
N_obs   = 5                              # only 5 measurements!
t_obs   = np.linspace(0.2, T*0.6, N_obs)  # measurements in first 60% of period
noise   = np.random.normal(0, 0.05, N_obs)  # 5% noise
x_obs   = x_exact(t_obs) + noise

print(f'\nSparse observations: {N_obs} noisy points')
print(f'  t_obs = {t_obs.round(3)}')
print(f'  x_obs = {x_obs.round(3)}')

In [ ]:
# ── Visualize the problem ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(t_full, x_full, color=ACCENT, lw=2, label='Analytical solution x(t) = cos(ωt)', zorder=2)
ax.scatter(t_obs, x_obs, color=WARM, s=80, zorder=5,
           label=f'{N_obs} noisy measurements (what we have)', edgecolors='white', lw=0.5)
ax.fill_between([t_obs.max(), T], -1.3, 1.3,
                color='#21262d', alpha=0.6, label='Unknown region (no data)')
ax.axhline(0, color='white', lw=0.5, ls='--')
ax.set_xlabel('Time t [s]')
ax.set_ylabel('Position x(t) [m]')
ax.set_title('The Challenge: Reconstruct x(t) from 5 sparse measurements\n'
             'mẍ + kx = 0  |  m=1 kg, k=4 N/m, ω=2 rad/s',
             fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(-1.3, 1.3)
plt.tight_layout()
plt.savefig('problem_setup.png', bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 🧠 3. Neural Network Architecture

A simple fully-connected network: **input = t**, **output = x(t)**

```
t  →  [64]  →  [64]  →  [64]  →  x(t)
       tanh     tanh     tanh
```

We use `tanh` activations because they are **infinitely differentiable** — essential for computing ẍ via automatic differentiation.

In [ ]:
class PINN(nn.Module):
    """
    Physics-Informed Neural Network for the harmonic oscillator.
    Input:  t  (time, scalar)
    Output: x  (position at time t)
    """
    def __init__(self, hidden=64, layers=3):
        super().__init__()

        # Build network: 1 → hidden → ... → hidden → 1
        net = [nn.Linear(1, hidden), nn.Tanh()]
        for _ in range(layers - 1):
            net += [nn.Linear(hidden, hidden), nn.Tanh()]
        net += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*net)

        # Xavier initialization — helps convergence
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, t):
        return self.net(t)


def compute_derivatives(model, t):
    """
    Compute x, ẋ, ẍ using automatic differentiation.
    This is the key operation that makes PINNs possible.
    """
    t = t.requires_grad_(True)

    x = model(t)                                           # x(t)

    # First derivative: ẋ = dx/dt
    dx_dt = torch.autograd.grad(
        x, t,
        grad_outputs=torch.ones_like(x),
        create_graph=True   # needed to differentiate again
    )[0]

    # Second derivative: ẍ = d²x/dt²
    d2x_dt2 = torch.autograd.grad(
        dx_dt, t,
        grad_outputs=torch.ones_like(dx_dt),
        create_graph=True
    )[0]

    return x, dx_dt, d2x_dt2


# Count parameters
model_pinn = PINN(hidden=64, layers=3).to(device)
model_nn   = PINN(hidden=64, layers=3).to(device)   # standard NN (no physics)
n_params = sum(p.numel() for p in model_pinn.parameters())
print(f'✅ Network architecture: 1 → 64 → 64 → 64 → 1')
print(f'   Total parameters: {n_params}')
print(f'   Activation: tanh (infinitely differentiable)')

## 🏋️ 4. Training

We train **two models** side by side:
- **Standard NN:** only $\mathcal{L}_{data}$ (no physics)
- **PINN:** $\mathcal{L}_{data} + \lambda \cdot \mathcal{L}_{physics}$

In [ ]:
# ── Convert data to tensors ───────────────────────────────────────────────────
t_obs_t = torch.tensor(t_obs, dtype=torch.float32).unsqueeze(1).to(device)
x_obs_t = torch.tensor(x_obs, dtype=torch.float32).unsqueeze(1).to(device)

# Collocation points: dense grid where we enforce the physics law
# (we don't need measurements here — just times where mẍ + kx = 0 must hold)
N_colloc = 200
t_colloc = torch.linspace(0, T, N_colloc).unsqueeze(1).to(device)

# Full time grid for evaluation
t_eval = torch.tensor(t_full, dtype=torch.float32).unsqueeze(1).to(device)

# ── Training function ─────────────────────────────────────────────────────────
def train_model(model, use_physics=True, epochs=8000, lambda_physics=1.0, lr=1e-3):
    """
    Train a PINN or standard NN.

    use_physics   : if True → PINN, if False → standard NN
    lambda_physics: weight of the physics loss term
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2000, gamma=0.5)

    loss_history       = []
    loss_data_history  = []
    loss_phys_history  = []

    model.train()
    for epoch in range(epochs + 1):

        optimizer.zero_grad()

        # ── Data loss: match the sparse observations ──────────────────────────
        x_pred_obs = model(t_obs_t)
        loss_data  = torch.mean((x_pred_obs - x_obs_t) ** 2)

        # ── Physics loss: mẍ + kx = 0 must hold at collocation points ─────────
        if use_physics:
            x_c, _, x_tt = compute_derivatives(model, t_colloc.clone())
            # Residual of the ODE: should be zero everywhere
            residual   = m * x_tt + k * x_c
            loss_phys  = torch.mean(residual ** 2)
        else:
            loss_phys = torch.tensor(0.0)

        # ── Total loss ────────────────────────────────────────────────────────
        loss = loss_data + lambda_physics * loss_phys

        loss.backward()
        optimizer.step()
        scheduler.step()

        loss_history.append(loss.item())
        loss_data_history.append(loss_data.item())
        loss_phys_history.append(loss_phys.item())

        if epoch % 1000 == 0:
            label = 'PINN' if use_physics else 'NN  '
            print(f'  [{label}] Epoch {epoch:5d} | '
                  f'L_total={loss.item():.6f} | '
                  f'L_data={loss_data.item():.6f} | '
                  f'L_phys={loss_phys.item():.6f}')

    return loss_history, loss_data_history, loss_phys_history


print('Training Standard NN (no physics)...')
hist_nn_total, hist_nn_data, _ = train_model(
    model_nn, use_physics=False, epochs=8000
)

print('\nTraining PINN (with physics)...')
hist_pinn_total, hist_pinn_data, hist_pinn_phys = train_model(
    model_pinn, use_physics=True, epochs=8000, lambda_physics=1.0
)

print('\n✅ Training complete!')

## 📊 5. Results & Comparison

In [ ]:
# ── Get predictions ───────────────────────────────────────────────────────────
model_pinn.eval()
model_nn.eval()

with torch.no_grad():
    x_pred_pinn = model_pinn(t_eval).cpu().numpy().flatten()
    x_pred_nn   = model_nn(t_eval).cpu().numpy().flatten()

# ── Errors ────────────────────────────────────────────────────────────────────
rmse_pinn = np.sqrt(np.mean((x_pred_pinn - x_full)**2))
rmse_nn   = np.sqrt(np.mean((x_pred_nn   - x_full)**2))

# Error in the unknown region only (after last observation)
t_unknown_mask = t_full > t_obs.max()
rmse_pinn_unk  = np.sqrt(np.mean((x_pred_pinn[t_unknown_mask] - x_full[t_unknown_mask])**2))
rmse_nn_unk    = np.sqrt(np.mean((x_pred_nn[t_unknown_mask]   - x_full[t_unknown_mask])**2))

print('=== Results ===')
print(f'  Global RMSE  — Standard NN : {rmse_nn:.4f} m')
print(f'  Global RMSE  — PINN        : {rmse_pinn:.4f} m')
print(f'  Unknown region RMSE — NN   : {rmse_nn_unk:.4f} m')
print(f'  Unknown region RMSE — PINN : {rmse_pinn_unk:.4f} m')
print(f'  Improvement: {(rmse_nn_unk - rmse_pinn_unk)/rmse_nn_unk * 100:.1f}% better in unknown region')

In [ ]:
# ── Main comparison plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('PINN vs Standard NN — Simple Harmonic Oscillator (mẍ + kx = 0)',
             fontsize=13, fontweight='bold')

# (a) Standard NN prediction
ax = axes[0, 0]
ax.plot(t_full, x_full,      color='white',  lw=1.5, ls='--', label='Exact solution', alpha=0.6)
ax.plot(t_full, x_pred_nn,   color=WARM,     lw=2,   label=f'Standard NN (RMSE={rmse_nn:.3f})')
ax.scatter(t_obs, x_obs, color=YELLOW, s=70, zorder=5, label='Observations', edgecolors='white', lw=0.5)
ax.axvline(t_obs.max(), color='#8b949e', ls=':', lw=1)
ax.set_title('(a) Standard NN — No Physics')
ax.set_xlabel('t [s]'); ax.set_ylabel('x(t) [m]')
ax.legend(fontsize=8); ax.set_ylim(-1.5, 1.5)

# (b) PINN prediction
ax = axes[0, 1]
ax.plot(t_full, x_full,      color='white',  lw=1.5, ls='--', label='Exact solution', alpha=0.6)
ax.plot(t_full, x_pred_pinn, color=GREEN,    lw=2,   label=f'PINN (RMSE={rmse_pinn:.3f})')
ax.scatter(t_obs, x_obs, color=YELLOW, s=70, zorder=5, label='Observations', edgecolors='white', lw=0.5)
ax.axvline(t_obs.max(), color='#8b949e', ls=':', lw=1)
ax.set_title('(b) PINN — Physics Enforced')
ax.set_xlabel('t [s]'); ax.set_ylabel('x(t) [m]')
ax.legend(fontsize=8); ax.set_ylim(-1.5, 1.5)

# (c) Absolute error comparison
ax = axes[1, 0]
ax.semilogy(t_full, np.abs(x_pred_nn   - x_full), color=WARM,  lw=1.5, label='Standard NN error')
ax.semilogy(t_full, np.abs(x_pred_pinn - x_full), color=GREEN, lw=1.5, label='PINN error')
ax.axvline(t_obs.max(), color='#8b949e', ls=':', lw=1, label='Last observation')
ax.set_xlabel('t [s]'); ax.set_ylabel('|error| [m]  (log scale)')
ax.set_title('(c) Absolute Error Comparison')
ax.legend(fontsize=8)

# (d) Training loss history
ax = axes[1, 1]
epochs_range = range(len(hist_pinn_total))
ax.semilogy(epochs_range, hist_nn_total,    color=WARM,   lw=1.5, label='NN total loss')
ax.semilogy(epochs_range, hist_pinn_total,  color=GREEN,  lw=1.5, label='PINN total loss')
ax.semilogy(epochs_range, hist_pinn_data,   color=ACCENT, lw=1,   ls='--', label='PINN data loss')
ax.semilogy(epochs_range, hist_pinn_phys,   color=PURPLE, lw=1,   ls='--', label='PINN physics loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (log scale)')
ax.set_title('(d) Training Loss History')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('pinn_results.png', bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 🔬 6. Physics Verification — Does the PINN Actually Satisfy mẍ + kx = 0?

In [ ]:
# Compute the ODE residual for both models
t_verify = torch.linspace(0, T, 500).unsqueeze(1).to(device)

model_pinn.train()  # need grad
model_nn.train()

x_p,  _, x_pp_pinn = compute_derivatives(model_pinn, t_verify.clone())
x_nn, _, x_pp_nn   = compute_derivatives(model_nn,   t_verify.clone())

with torch.no_grad():
    residual_pinn = (m * x_pp_pinn + k * x_p).cpu().numpy().flatten()
    residual_nn   = (m * x_pp_nn   + k * x_nn).cpu().numpy().flatten()
    t_v_np        = t_verify.cpu().numpy().flatten()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t_v_np, residual_nn,   color=WARM,  lw=1.5, label=f'Standard NN: mẍ + kx  (should be 0)', alpha=0.8)
ax.plot(t_v_np, residual_pinn, color=GREEN, lw=1.5, label=f'PINN:        mẍ + kx  (should be 0)')
ax.axhline(0, color='white', lw=1, ls='--', label='Perfect: residual = 0')
ax.fill_between(t_v_np, -0.05, 0.05, color='white', alpha=0.04)
ax.set_xlabel('t [s]')
ax.set_ylabel('ODE Residual: mẍ + kx')
ax.set_title('Physics Verification: How well does each model satisfy mẍ + kx = 0?',
             fontweight='bold')
ax.legend(fontsize=9)

print(f'Mean |residual| — Standard NN : {np.abs(residual_nn).mean():.6f}')
print(f'Mean |residual| — PINN        : {np.abs(residual_pinn).mean():.6f}')
print(f'Physics satisfaction improvement: {np.abs(residual_nn).mean() / np.abs(residual_pinn).mean():.1f}×')

plt.tight_layout()
plt.savefig('ode_residual.png', bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 🧪 7. Experiment: What Happens if We Change λ?

In [ ]:
# ── Effect of physics weight λ ────────────────────────────────────────────────
# λ=0   → pure data fitting (= standard NN)
# λ=0.1 → weak physics
# λ=1.0 → balanced
# λ=10  → strong physics

lambdas = [0.0, 0.1, 1.0, 10.0]
results = {}

print('Training models with different λ values...')
for lam in lambdas:
    torch.manual_seed(42)
    m_tmp = PINN(hidden=64, layers=3).to(device)
    train_model(m_tmp, use_physics=(lam > 0),
                epochs=5000, lambda_physics=lam, lr=1e-3)
    m_tmp.eval()
    with torch.no_grad():
        pred = m_tmp(t_eval).cpu().numpy().flatten()
    rmse = np.sqrt(np.mean((pred - x_full)**2))
    rmse_unk = np.sqrt(np.mean((pred[t_unknown_mask] - x_full[t_unknown_mask])**2))
    results[lam] = {'pred': pred, 'rmse': rmse, 'rmse_unk': rmse_unk}
    print(f'  λ={lam:5.1f} → RMSE={rmse:.4f} | RMSE (unknown)={rmse_unk:.4f}')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(lambdas), figsize=(15, 4), sharey=True)
fig.suptitle('Effect of Physics Weight λ on PINN Performance', fontweight='bold', fontsize=12)

colors_lam = [WARM, YELLOW, GREEN, ACCENT]
for ax, lam, col in zip(axes, lambdas, colors_lam):
    ax.plot(t_full, x_full, color='white', lw=1.5, ls='--', alpha=0.5, label='Exact')
    ax.plot(t_full, results[lam]['pred'], color=col, lw=2,
            label=f'λ={lam}')
    ax.scatter(t_obs, x_obs, color=YELLOW, s=50, zorder=5, edgecolors='white', lw=0.5)
    ax.axvline(t_obs.max(), color='#8b949e', ls=':', lw=1)
    ax.set_title(f'λ = {lam}\nRMSE={results[lam]["rmse"]:.3f}', fontsize=10)
    ax.set_xlabel('t [s]')
    ax.set_ylim(-1.8, 1.8)
    if ax == axes[0]:
        ax.set_ylabel('x(t) [m]')

plt.tight_layout()
plt.savefig('lambda_experiment.png', bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 📋 8. Summary

In [ ]:
print('=' * 58)
print('  PINN — SIMPLE HARMONIC OSCILLATOR — KEY FINDINGS')
print('=' * 58)
print()
print(f'  Physical system  : mẍ + kx = 0')
print(f'  Parameters       : m={m} kg, k={k} N/m, ω={omega:.3f} rad/s')
print(f'  Training data    : only {N_obs} noisy observations')
print()
print('  PERFORMANCE (full domain):')
print(f'    Standard NN RMSE : {rmse_nn:.4f} m')
print(f'    PINN RMSE        : {rmse_pinn:.4f} m')
print()
print('  PERFORMANCE (unknown region — no data):')
print(f'    Standard NN RMSE : {rmse_nn_unk:.4f} m')
print(f'    PINN RMSE        : {rmse_pinn_unk:.4f} m')
print()
print('  PHYSICS SATISFACTION (mean |mẍ + kx|):')
print(f'    Standard NN      : {np.abs(residual_nn).mean():.6f}')
print(f'    PINN             : {np.abs(residual_pinn).mean():.6f}')
print()
print('  KEY INSIGHT:')
print('    The PINN uses physical knowledge (Newton\'s 2nd Law)')
print('    to extrapolate correctly beyond the observed region.')
print('    Standard NN fails because it has no physics constraint.')
print('=' * 58)

---

## 💡 Things to Try

1. **Change N_obs** from 5 to 3 or to 10 — how does the PINN hold up?
2. **Change λ** — what's the optimal value?
3. **Add damping** — modify the ODE to mẍ + γẋ + kx = 0 and update the physics loss
4. **Increase noise** from 0.05 to 0.2 — does the PINN remain robust?

